# Random vs BFS 기준선

## Experiment Goal

- LangGraph 실행기에서 Random과 BFS 기준선이 같은 cohort를 사용하는지 확인한다.
- 저장 출력은 없다. 위에서 아래로 실행해 현재 결과를 만든다.

## Context & Methods

두 기준선을 같은 내장 레벨과 seed 조합에서 실행한다. 환경의 `max_steps`는 모든 실행에 동일하게 적용한다.

### Key Assumptions

- 현재 비교 범위는 `tiny-push`, `tiny-walk`다.
- BFS는 primitive action 최단 경로를 완전 탐색한다.
- 실행 시간에는 LangGraph 초기화·계획·검증·실행이 포함된다.

### 1. Setup

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd

from sokoban_agent.env import SokobanEnv
from sokoban_agent.evaluation import (
    run_benchmark,
    summarize_by_planner,
)
from sokoban_agent.planning import BFSPlanner, RandomPlanner

LEVEL_IDS = ['tiny-push', 'tiny-walk']
SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MAX_STEPS = 40

## Data

### 2. Run identical cases

In [ ]:
env = SokobanEnv(max_steps=MAX_STEPS)
try:
    results = run_benchmark(
        env,
        [RandomPlanner(), BFSPlanner()],
        level_ids=LEVEL_IDS,
        seeds=SEEDS,
    )
finally:
    env.close()

results_df = pd.DataFrame.from_records(
    asdict(result) for result in results
)
results_df.head(8)

## Results

### 3. Compare metrics

In [ ]:
summary_df = pd.DataFrame.from_records(
    asdict(summary) for summary in summarize_by_planner(results)
).set_index('planner_name')
summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
summary_df['success_rate'].plot.bar(
    ax=axes[0], ylim=(0, 1), title='Success rate'
)
summary_df['mean_actions'].plot.bar(
    ax=axes[1], title='Mean actions per episode'
)
for axis in axes:
    axis.set_xlabel('Graph planner')
    axis.tick_params(axis='x', rotation=0)
fig.tight_layout()
plt.show()

### 4. Validate the comparison cohort

In [ ]:
case_sets = {
    planner_name: set(
        zip(group['level_id'], group['seed'], strict=True)
    )
    for planner_name, group in results_df.groupby('planner_name')
}
assert case_sets['graph:random'] == case_sets['graph:bfs']
assert len(case_sets['graph:bfs']) == len(LEVEL_IDS) * len(SEEDS)
assert summary_df.loc['graph:bfs', 'success_rate'] == 1.0
case_sets

## Takeaways

실행 후 위 집계표와 검증 셀을 기준선 결과로 사용한다. 이 실험은 작은 내장 레벨만 다룬다.